# 🌌 Colab Router — Shift 2: Secondary Account (account2@example.com)
### 🕒 Dedicated Slot: **Shift 2 (16:00 - 00:00 UTC)** | 👤 Account: `account2@example.com`

Turn Google Colab (Tesla T4 GPU) into an **OpenAI-Compatible Streaming Inference Server** with 100% GPU VRAM offload.

### 📌 IMPORTANT — Multi-Account Setup Instructions:
1. **Check Logged In Account:** Ensure this tab is running under **`account2@example.com`** (check the top-right profile avatar).
2. **Save to Drive:** Click `File` ➔ `Save a copy in Drive` inside this account to keep your own independent notebook.
3. **Verify T4 GPU:** Click `Runtime` ➔ `Change runtime type` ➔ Ensure **T4 GPU** is selected (Python 3).
4. **Enter Tokens:** Add your `NGROK_AUTHTOKEN` in Cell 2 (or save in Colab 🔑 Secrets tab).
5. **Run All:** Press `Ctrl+F9` or click `Runtime` ➔ `Run all`.

---


In [ ]:
# ==============================================================================
# 🛡️ STEP 1: HARDWARE PRE-FLIGHT CHECK (Strict T4 GPU Verification)
# ==============================================================================
import sys, torch

print("🔍 Checking Hardware Accelerator...")
if not torch.cuda.is_available():
    error_banner = """
    ❌❌❌ FATAL: NO GPU DETECTED (Running on CPU) ❌❌❌
    Running 27B LLM on CPU will be 100x slower or crash with Out-Of-Memory.
    
    👉 HOW TO FIX RIGHT NOW:
       1. In top menu, click: Runtime ➔ Change runtime type
       2. Under Hardware accelerator, select 'T4 GPU'
       3. Click 'Save'
       4. Re-run notebook: Runtime ➔ Run all (Ctrl+F9)
    """
    print(error_banner, file=sys.stderr)
    raise RuntimeError("T4 GPU Required! Please enable GPU in Colab Runtime settings.")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"🚀 ACTIVE ACCELERATOR: {gpu_name} ({gpu_mem:.1f} GB VRAM)")
print("✅ GPU Gatekeeper Passed for account2@example.com! Ready for 100% VRAM offload.\n")


In [ ]:
# ==============================================================================
# ⚙️ STEP 2: CONFIGURATION & CREDENTIALS
# ==============================================================================
import os

# 🔑 Read secrets independently from Colab Secrets tab (key icon on left)
def get_secret(key, default=None):
    try:
        from google.colab import userdata
        val = userdata.get(key)
        return val if val is not None else default
    except Exception:
        return default

ngrok_token = get_secret('NGROK_AUTHTOKEN') or os.environ.get('NGROK_AUTHTOKEN', '')
ngrok_domain = get_secret('NGROK_DOMAIN') or os.environ.get('NGROK_DOMAIN', '')
api_key = get_secret('GDQ_API_KEY') or os.environ.get('GDQ_API_KEY', '')
passkey = get_secret('GDQ_PASSKEY') or os.environ.get('GDQ_PASSKEY', '')

# If NGROK_AUTHTOKEN is missing, prompt interactively rather than using a broken placeholder
if not ngrok_token or ngrok_token == 'YOUR_NGROK_AUTHTOKEN_HERE':
    import getpass
    print('⚠️ NGROK_AUTHTOKEN not found in Colab Secrets tab.')
    ngrok_token = getpass.getpass('Enter your Ngrok Authtoken: ').strip()

if not api_key and not passkey:
    import getpass
    print('🔑 No API Key or Passkey configured in Colab Secrets.')
    passkey = getpass.getpass("Enter a Passkey for gateway access (or press Enter for 'sovereign-pass'): ").strip() or 'sovereign-pass'

os.environ['NGROK_AUTHTOKEN'] = ngrok_token
os.environ['NGROK_DOMAIN'] = ngrok_domain or ''
os.environ['GDQ_API_KEY'] = api_key or ''
os.environ['GDQ_PASSKEY'] = passkey or ''

# Model Selection (Default: Qwen 3.8 27B Uncensored IQ4_XS - 14.3 GB)
# For large context headroom or if T4 VRAM is tight, IQ2_M (10.6 GB) can be set in HF_MODEL_URL
os.environ['HF_MODEL_URL'] = os.environ.get(
    'HF_MODEL_URL',
    'https://huggingface.co/JonathanColetti/Qwen3.8-27B-Uncensored-GGUF/resolve/main/Qwen3.8-27B-Uncensored-IQ4_XS.gguf'
)
os.environ['MODEL_FILENAME'] = os.environ.get('MODEL_FILENAME', 'Qwen3.8-27B-Uncensored-IQ4_XS.gguf')
os.environ['OLLAMA_MODEL_TAG'] = os.environ.get('OLLAMA_MODEL_TAG', 'qwen3.8-27b-uncensored')
os.environ['NUM_CTX'] = os.environ.get('NUM_CTX', '4096')
os.environ['NUM_PREDICT'] = os.environ.get('NUM_PREDICT', '2048')

print('✅ Configuration loaded successfully!')


In [ ]:
# ==============================================================================
# 🚀 STEP 3: LAUNCH ENGINE & HIGHWAY ROUTER
# ==============================================================================
import os, subprocess

GITHUB_REPO_URL = "https://github.com/agenticversestudio/colab-router.git"
repo_dir = "/content/colab-router"
scripts_ready = False

print(f"📦 Checking Colab Router engine repository...")
if os.path.exists(repo_dir):
    print("🔄 Pulling latest engine code...")
    subprocess.run(["git", "-C", repo_dir, "pull"], capture_output=True, text=True)
    os.system(f"cp {repo_dir}/colab/colab_payload.py /content/colab_payload.py")
    os.system(f"cp {repo_dir}/colab/hybrid_server.py /content/hybrid_server.py")
    scripts_ready = True
else:
    print(f"📦 Cloning from {GITHUB_REPO_URL}...")
    res = subprocess.run(["git", "clone", GITHUB_REPO_URL, repo_dir], capture_output=True, text=True)
    if res.returncode == 0:
        os.system(f"cp {repo_dir}/colab/colab_payload.py /content/colab_payload.py")
        os.system(f"cp {repo_dir}/colab/hybrid_server.py /content/hybrid_server.py")
        scripts_ready = True
    else:
        print(f"⚠️ Clone failed: {res.stderr.strip()}")

# Fallback to /content files if uploaded directly
if not scripts_ready and os.path.exists("/content/colab_payload.py") and os.path.exists("/content/hybrid_server.py"):
    scripts_ready = True

if not scripts_ready:
    raise FileNotFoundError("Unable to locate colab_payload.py. Please upload files to /content or check network access.")

print("\n🔥 Starting Sovereign Colab Router Autonomous Engine on Tesla T4...")
res = subprocess.run(["python3", "/content/colab_payload.py"])
if res.returncode != 0:
    raise RuntimeError(f"❌ Colab Router engine failed with exit code {res.returncode}. See log above.")
